# Colab DeBERTa Training

Imports the existing DeBERTa training module and runs it against Drive-backed splits and persistent outputs.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/data/llm-gatekeeping'
SPLITS_DIR = f'{DRIVE_ROOT}/data/processed/splits'
ARTIFACTS_DIR = f'{DRIVE_ROOT}/artifacts/deberta_classifier'
PREDICTIONS_DIR = f'{DRIVE_ROOT}/data/processed/predictions'
REPORTS_DIR = f'{DRIVE_ROOT}/reports/deberta_classifier'

REPO_URL = 'https://github.com/noamdwc/llm-gatekeeping.git'
REPO_DIR = '/content/llm-gatekeeping'
WANDB_PROJECT = 'llm-gatekeeping'
WANDB_RUN_NAME = 'deberta-colab'
USE_WANDB = True
DEVICE = 'cuda'
NUM_EPOCHS = 5
BATCH_SIZE = 8
LEARNING_RATE = 5e-6

In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Using existing repo at {REPO_DIR}')

os.chdir(REPO_DIR)
print('Repo:', os.getcwd())


!git checkout zero_shot_nlp_attack # Move to the current branch that we work on

In [ ]:
%pip install -r requirements.txt

In [ ]:
if USE_WANDB:
    import wandb
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))

In [ ]:
from pathlib import Path

import pandas as pd
import torch

required_splits = ['train', 'val']
optional_splits = ['test', 'unseen_val', 'unseen_test', 'safeguard_test']
required_columns = {'modified_sample', 'label_binary'}
valid_labels = {'benign', 'adversarial'}

if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Requested CUDA, but Colab GPU is unavailable. Enable Runtime > Change runtime type > GPU.')

splits_path = Path(SPLITS_DIR)
for split in required_splits:
    path = splits_path / f'{split}.parquet'
    if not path.exists():
        raise FileNotFoundError(f'Missing required split: {path}')

for split in required_splits + optional_splits:
    path = splits_path / f'{split}.parquet'
    if not path.exists():
        print(f'Skipping optional missing split: {path}')
        continue
    df = pd.read_parquet(path)
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f'{path} missing required columns: {sorted(missing)}')
    labels = set(df['label_binary'].dropna().unique())
    invalid = labels - valid_labels
    if invalid or df['label_binary'].isna().any():
        raise ValueError(f'{path} has invalid label_binary values: {sorted(invalid)}')
    print(f'{split}: {len(df):,} rows, labels={df.label_binary.value_counts().to_dict()}')

for out_dir in [ARTIFACTS_DIR, PREDICTIONS_DIR, REPORTS_DIR]:
    p = Path(out_dir)
    p.mkdir(parents=True, exist_ok=True)
    probe = p / '.write_test'
    probe.write_text('ok')
    probe.unlink()
    print(f'Writable: {p}')

In [ ]:
from src.cli.deberta_classifier import parse_args, run_deberta

argv = [
    '--research',
    '--splits-dir', SPLITS_DIR,
    '--artifacts-dir', ARTIFACTS_DIR,
    '--predictions-dir', PREDICTIONS_DIR,
    '--reports-dir', REPORTS_DIR,
    '--wandb-project', WANDB_PROJECT,
    '--wandb-run-name', WANDB_RUN_NAME,
    '--device', DEVICE,
    '--num-epochs', str(NUM_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
]
if not USE_WANDB:
    argv.append('--no-wandb')

print('run_deberta args:', ' '.join(argv))
args = parse_args(argv)
run_deberta(args)